# 20 · «Серьёзная тревога» с окном подтверждения 30 МИН вместо 6 ЧАСОВ

Текущее определение (§9): подтверждённое задымление = **≥2 дымовых каналов
объекта за 6-часовой бакет** ИЛИ ручной ИПР. Окно 6ч слишком широкое — два ИПР,
задымившихся в разных концах дня, уже «подтверждение». Проверяем окно **30 минут**
по точному времени: эпизод «серьёзный», если за ≤30 мин на объекте сработали
≥2 дымовых канала (или был ручной ИПР).


In [1]:
%matplotlib inline
import sys
import pathlib

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd

import data_utils as du
import features as fe
import tte_pipeline as tte
from tte_experiments import survival_metrics, pr_at_horizon, fit_discrete_hazard


DATASET = du.find_dataset_dir()
ref = du.load_ref_channels()[["ид_канала_данных", "тип_датчика", "ид_объект"]]
ref = ref.dropna(subset=["ид_объект"])
smoke_ch = set(ref.loc[ref["тип_датчика"] == "Датчик дыма", "ид_канала_данных"])
manual_ch = set(ref.loc[ref["тип_датчика"] == "Ручной извещатель", "ид_канала_данных"])
ch_to_obj = dict(zip(ref["ид_канала_данных"].astype(str), ref["ид_объект"].astype(str)))
hour_ns = fe.BUCKET_HOURS * 3600 * 10 ** 9
GAP_NS = 30 * 60 * 10 ** 9
print("дымовых:", len(smoke_ch), "| ручных:", len(manual_ch))


дымовых: 4337 | ручных: 130


In [2]:
# 1) Проход: события «Обнаружен дым» / «Рычаг сдернут» (~5-10 мин)
parts = []
for fp in du.journal_files():
    for ch in du.read_chunks(fp, cols=["ид_канала_данных", "дата", "время",
                                       "значение_датчика"], chunksize=500_000):
        m = ch["ид_канала_данных"].isin(smoke_ch | manual_ch)
        if not m.any():
            continue
        ch = ch[m]
        v = ch["значение_датчика"].fillna("<нет>").astype(str)
        ev = (v == "Обнаружен дым") | (v == "Рычаг сдернут")
        if not ev.any():
            continue
        ch = ch[ev]
        ts = pd.to_datetime(ch["дата"] + " " + ch["время"].fillna("00:00:00"),
                            format="%Y-%m-%d %H:%M:%S", errors="coerce")
        ch = ch[ts.notna()]
        ts = ts[ts.notna()]
        ids = ch["ид_канала_данных"].astype(str).to_numpy()
        tbl = pd.DataFrame({"ид": ids, "t": ts.astype("datetime64[ns]").astype("int64").to_numpy()})
        tbl["об"] = tbl["ид"].map(ch_to_obj)
        tbl["тип"] = np.where(tbl["ид"].isin(smoke_ch), "дым", "ручной")
        parts.append(tbl.dropna(subset=["об"]))
    print(f"  {fp.name}: n={sum(len(p) for p in parts):,}", flush=True)

EV = pd.concat(parts, ignore_index=True)
EV.to_parquet(DATASET / "fire_raw_events_30m.parquet")
print("всего событий:", len(EV), "| дым:", int((EV["тип"] == "дым").sum()),
      "| ручной:", int((EV["тип"] == "ручной").sum()))


  ext-journal-2019.csv: n=5,228


  ext-journal-2020.csv: n=16,762


  ext-journal-2021.csv: n=28,384


  ext-journal-2022.csv: n=53,878


  ext-journal-2023.csv: n=89,861


  ext-journal-2024.csv: n=120,235


  ext-journal-2025.csv: n=148,543


  ext-journal-2026.csv: n=164,477


всего событий: 164477 | дым: 164288 | ручной: 189


In [3]:
# 2) Эпизоды по объекту (разрыв <=30 мин); серьёзный: >=2 дымовых канала ИЛИ ручной
def detect2(ev_ob):
    ev_ob = ev_ob.sort_values("t").reset_index(drop=True)
    t = ev_ob["t"].to_numpy()
    gid = np.cumsum(np.concatenate([[True], t[1:] - t[:-1] > GAP_NS])) - 1
    ev2 = ev_ob.copy(); ev2["gid"] = gid
    cnt = ev2[ev2["тип"] == "дым"].groupby("gid")["ид"].nunique().rename("n_dym")
    agg = ev2.groupby("gid").agg(t0=("t", "min"), t1=("t", "max"),
                                 n_manual=("тип", lambda s: int((s == "ручной").sum())))
    agg = agg.join(cnt).fillna({"n_dym": 0})
    agg["serious"] = (agg["n_dym"] >= 2) | (agg["n_manual"] > 0)
    agg["бакет"] = (agg["t0"] // hour_ns).astype(np.int64)
    return agg[agg["serious"]]

EPISODES = []
for об, grp in EV.groupby("об"):
    d = detect2(grp)
    if len(d):
        d = d.reset_index().rename(columns={"index": "gid"})
        d["об"] = об
        EPISODES.append(d)
EP = pd.concat(EPISODES, ignore_index=True) if EPISODES else pd.DataFrame()
EP.to_parquet(DATASET / "fire_episodes_30m.parquet")
print("серьёзных эпизодов (30 мин):", len(EP),
      "| ручных:", int((EP["n_manual"] > 0).sum()),
      "| дымовых (>=2 ИПР):", int((EP["n_manual"] == 0).sum()))


серьёзных эпизодов (30 мин): 2303 | ручных: 53 | дымовых (>=2 ИПР): 2250


In [4]:
# 3) Колонка серьёзных_30м на панели: бакет с стартом эпизода у объекта
ep = pd.read_parquet(DATASET / "fire_episodes_30m.parquet")
ep["об"] = ep["об"].astype(str)
start_map = ep[["об", "бакет"]].drop_duplicates().rename(columns={"об": "ид_объект"})

panel = pd.read_csv(DATASET / "subdaily_panel_fire6h.csv",
                    dtype={"ид_канала_данных": str, "ид_объект": str})
panel = panel.merge(start_map, on=["ид_объект", "бакет"], how="left", indicator="m30")
panel["серьёзных_30м"] = (panel["m30"] == "both").astype(np.int8)
panel = panel.drop(columns=["m30"])
print("база события 30м на панели: {:.3f}%".format(panel["серьёзных_30м"].mean() * 100))
print("база события 6ч на панели:  {:.3f}%".format(panel["серьёзных"].mean() * 100))
print("эпизоды 30м, не увиденные 6ч-событием:",
      int(((panel["серьёзных_30м"] == 1) & (panel["серьёзных"] == 0)).sum()))


база события 30м на панели: 11.801%
база события 6ч на панели:  8.522%
эпизоды 30м, не увиденные 6ч-событием: 58542


In [5]:
# 4) Субъекты TTE (событие = старт серии серьёзных_30м) и обучение fire
panel = tte.refit_z_train_only(panel)
panel = tte.add_series_context(panel, event_col="серьёзных_30м")
subjects = tte.add_tte_labels(panel)
subjects["ид_объект_code"] = subjects["ид_объект"].astype("category").cat.codes
subjects["тип_датчика_code"] = subjects["тип_датчика"].astype("category").cat.codes
tr, va, ho = tte.split_subjects(subjects)
dt = pd.to_datetime(tr["дата"])
cl = ~tr["аномально"].to_numpy()
ev = tr["event_flag"].fillna(0).to_numpy(); od = tr["obs_days"].to_numpy()
print("train база серьёзных_30м: <=1д {:.3f}% | <=7д {:.3f}% | <=30д {:.3f}%".format(
    ((ev == 1) & (od <= 1.01)).mean() * 100,
    ((ev == 1) & (od <= 7.01)).mean() * 100,
    ((ev == 1) & (od <= 30.01)).mean() * 100))
print("событий-серий 30м в train (D<=1д):", int(((ev == 1) & (od <= 1.01)).sum()))


train база серьёзных_30м: <=1д 1.918% | <=7д 9.423% | <=30д 22.414%
событий-серий 30м в train (D<=1д): 16341


In [6]:
def run(X_cols):
    trs = tte.sample_subjects(tr, 15000, 10000, 10000, seed=42)
    vas = tte.sample_subjects(va, 2000, 2000, 2000, seed=43)
    hos = tte.sample_holdout(ho, 20000, seed=777)
    X_tr, y_tr = tte.expand_person_time(trs, X_cols, tte.HORIZON_BUCKETS)
    X_va, y_va = tte.expand_person_time(vas, X_cols, tte.HORIZON_BUCKETS)
    m = fit_discrete_hazard(X_tr, y_tr, X_va, y_va, iters=1000, lr=0.03,
                            depth=10, seed=42)
    surv_tr = tte.make_surv_struct(trs["event_flag"], trs["obs_days"])
    surv_ho = tte.make_surv_struct(hos["event_flag"], hos["obs_days"])
    S = np.cumprod(1 - m.predict_proba(
        tte.expand_person_time(hos, X_cols, tte.HORIZON_BUCKETS, fixed_horizon=True)[0]
        )[:, 1].reshape(-1, tte.HORIZON_BUCKETS), axis=1)
    risk = 1 - S[:, -1]
    met = survival_metrics(surv_tr, surv_ho, risk, S, tte.EVAL_TIMES)
    y24 = ((hos["event_flag"] == 1) & (hos["obs_days"] <= 1.01)).to_numpy(int)
    p24 = 1 - S[:, int(round(1.0 * tte.STEPS_PER_DAY)) - 1]
    pr = pr_at_horizon(y24, p24)
    return met, pr

X_cols = tte.subject_features(tr)
met30, pr30 = run(X_cols)
print(f"[серьёзная 30м] Uno-C={met30['uno_c']:.3f} | IBS={met30['ibs']:.3f} "
      f"| P(24ч) PR-AUC={pr30['pr_auc']:.3f} (база {pr30['base']*100:.2f}%)")
print()
print("Для сравнения (событие 6ч, те же дефолты): Uno-C=0.792, P(24ч) PR-AUC=0.421, база 1.53%")


0:	learn: 0.6006902	test: 0.6002544	best: 0.6002544 (0)	total: 1.2s	remaining: 19m 58s


100:	learn: 0.0091204	test: 0.0097670	best: 0.0097670 (100)	total: 1m 45s	remaining: 15m 41s


200:	learn: 0.0070601	test: 0.0095751	best: 0.0095620 (129)	total: 3m 31s	remaining: 13m 58s


300:	learn: 0.0063060	test: 0.0096852	best: 0.0095620 (129)	total: 5m 19s	remaining: 12m 23s


400:	learn: 0.0058660	test: 0.0097406	best: 0.0095620 (129)	total: 7m 3s	remaining: 10m 32s


Stopped by overfitting detector  (300 iterations wait)

bestTest = 0.009561962446
bestIteration = 129

Shrink model to first 130 iterations.


[серьёзная 30м] Uno-C=0.696 | IBS=0.080 | P(24ч) PR-AUC=0.120 (база 3.24%)

Для сравнения (событие 6ч, те же дефолты): Uno-C=0.792, P(24ч) PR-AUC=0.421, база 1.53%


### Выводы
- Если `серьёзных_30м` даёт сопоставимые/лучшие метрики и при этом база ниже —
  значит окно 6ч добавляло «ложные подтверждения», и определение 30 мин честнее.
- Если метрики заметно хуже — значит 30-минутное подтверждение режет нужные
  события (задымления растянуты во времени), и окно 6ч, наоборот, ближе к
  реальности. Тогда вопрос в границах (например, 1–2 часа) — следующий тест.
